In [0]:
"""
Data Quality & Reconciliation Test Suite for Gold Medallion Layer.
Framework: pytest + PySpark
"""

import pytest
from pyspark.sql import SparkSession

@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder.getOrCreate()

def test_gold_fact_table_not_empty(spark):
    """Ensure fact_wikipedia_edits has ingested records."""
    count = spark.table("dbr_dev.wikimediademo_gold.fact_wikipedia_edits").count()
    assert count > 0, "Assertion Failed: fact_wikipedia_edits table is empty."

def test_dim_project_primary_keys_unique(spark):
    """Ensure surrogate primary keys in dim_project are strictly unique."""
    duplicates = (
        spark.table("dbr_dev.wikimediademo_gold.dim_project")
        .groupBy("project_key")
        .count()
        .filter("count > 1")
        .count()
    )
    assert duplicates == 0, f"Assertion Failed: Found {duplicates} duplicate keys in dim_project."

def test_referential_integrity_fact_to_dim_project(spark):
    """Ensure all facts map to a valid dimension key (No Orphan Records)."""
    orphan_count = spark.sql("""
        SELECT count(*) AS orphan_count
        FROM dbr_dev.wikimediademo_gold.fact_wikipedia_edits f
        LEFT JOIN dbr_dev.wikimediademo_gold.dim_project p 
            ON f.project_key = p.project_key
        WHERE p.project_key IS NULL
    """).collect()[0]["orphan_count"]
    
    assert orphan_count == 0, f"Assertion Failed: Found {orphan_count} orphan fact records without dimension mappings."

def test_silver_to_gold_reconciliation(spark):
    """Ensure 100% reconciliation matching between Silver stream source and Gold fact table."""
    silver_count = spark.table("dbr_dev.wikimediademo_silver.wikipedia_edits").count()
    gold_count = spark.table("dbr_dev.wikimediademo_gold.fact_wikipedia_edits").count()
    
    assert silver_count == gold_count, (
        f"Reconciliation Mismatch: Silver count ({silver_count}) != Gold fact count ({gold_count})"
    )

In [0]:
from pyspark.sql import SparkSession

active_spark = SparkSession.builder.getOrCreate()

tests = [
    test_gold_fact_table_not_empty,
    test_dim_project_primary_keys_unique,
    test_referential_integrity_fact_to_dim_project,
    test_silver_to_gold_reconciliation
]

print("=== Running Gold Layer Quality Tests ===")
for test in tests:
    try:
        test(active_spark)
        print(f"PASSED: {test.__name__}")
    except AssertionError as e:
        print(f"FAILED: {test.__name__} -> {e}")
    except Exception as e:
        print(f"ERROR:  {test.__name__} -> {e}")
print("========================================")